# This Week's Movie Recommender

A hybrid recommender that suggests movies for a user to watch **this week** by blending two signals:

1. **Personal taste** — a matrix-factorization collaborative filtering model (SVD) trained on the [MovieLens](https://grouplens.org/datasets/movielens/) `ml-latest-small` dataset (100,836 ratings, 610 users, ~9,700 movies), predicting how much a given user would rate a movie they haven't seen yet.
2. **This week's buzz** — a recency-weighted popularity score, so the picks aren't just "things you'd generically like" but skew toward what's getting attention *right now*.

The two are combined into a single **hybrid score** per candidate movie, controlled by a tunable weight `alpha`.

> **Note on "this week":** MovieLens is a static historical dataset (ratings run 1996–2018), so there's no real "currently trending" signal to pull from. To make the notebook self-contained and offline, the recency window is simulated by treating the most recent timestamp in the dataset as "now" — the trending logic itself (exponential decay + Bayesian-adjusted average) is exactly what you'd run against live rating/watch events in production. See the last section for notes on plugging in a real live feed (e.g. TMDb "trending this week").

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

DATA_DIR = Path("data/ml-latest-small")
RANDOM_STATE = 42

## 1. Load data

In [ ]:
ratings = pd.read_csv(DATA_DIR / "ratings.csv")
movies = pd.read_csv(DATA_DIR / "movies.csv")

ratings["timestamp"] = pd.to_datetime(ratings["timestamp"], unit="s")

print(f"{ratings['userId'].nunique()} users, {ratings['movieId'].nunique()} rated movies, {len(ratings)} ratings")
print(f"Ratings span {ratings['timestamp'].min().date()} to {ratings['timestamp'].max().date()}")
ratings.head()

In [ ]:
ratings["rating"].plot(kind="hist", bins=9, title="Rating distribution", edgecolor="white")
plt.xlabel("rating")
plt.show()

## 2. Personal taste model: matrix factorization (SVD)

Build a dense user x movie ratings matrix, mean-center each user's ratings (so the model learns relative preference rather than being thrown off by generous vs. harsh raters), and factorize it with truncated SVD. Reconstructing the matrix from the low-rank factors gives a predicted rating for *every* user/movie pair — including ones the user hasn't rated, which is exactly what we need for recommendations.

In [ ]:
train_df, test_df = train_test_split(ratings, test_size=0.2, random_state=RANDOM_STATE)

user_ids = ratings["userId"].unique()
movie_ids = ratings["movieId"].unique()
user_idx = {u: i for i, u in enumerate(user_ids)}
movie_idx = {m: i for i, m in enumerate(movie_ids)}

def build_matrix(df):
    mat = np.zeros((len(user_ids), len(movie_ids)))
    for row in df.itertuples():
        mat[user_idx[row.userId], movie_idx[row.movieId]] = row.rating
    return mat

train_matrix = build_matrix(train_df)
mask = train_matrix > 0

user_means = np.array([
    train_matrix[i, mask[i]].mean() if mask[i].any() else train_df["rating"].mean()
    for i in range(train_matrix.shape[0])
])

matrix_centered = np.where(mask, train_matrix - user_means.reshape(-1, 1), 0)

svd = TruncatedSVD(n_components=50, random_state=RANDOM_STATE)
user_factors = svd.fit_transform(matrix_centered)
item_factors = svd.components_

pred_matrix = user_factors @ item_factors + user_means.reshape(-1, 1)
pred_matrix = np.clip(pred_matrix, 0.5, 5.0)

print("Explained variance captured by 50 components:", svd.explained_variance_ratio_.sum().round(3))

### Evaluate: RMSE on held-out ratings

In [ ]:
def rmse_on(df, pred_matrix):
    errs = [
        (pred_matrix[user_idx[row.userId], movie_idx[row.movieId]] - row.rating) ** 2
        for row in df.itertuples()
        if row.userId in user_idx and row.movieId in movie_idx
    ]
    return np.sqrt(np.mean(errs))

print("Test RMSE:", round(rmse_on(test_df, pred_matrix), 4))

## 3. This week's buzz: recency-weighted trending score

For each movie, weight its recent ratings by an exponential decay based on how many days old they are (half-life of 14 days), then compute a Bayesian-adjusted average rating so that a movie with one 5-star rating this week doesn't outrank a movie with hundreds of consistently strong ratings.

In [ ]:
NOW = ratings["timestamp"].max()   # simulated "today" — swap for pd.Timestamp.now() with a live feed
WINDOW_DAYS = 90                    # lookback window feeding the pulse (production: last 7-30 days is plenty)
HALF_LIFE_DAYS = 14                 # how fast older activity stops mattering

recent = ratings[ratings["timestamp"] >= NOW - pd.Timedelta(days=WINDOW_DAYS)].copy()
recent["age_days"] = (NOW - recent["timestamp"]).dt.total_seconds() / 86400
recent["decay"] = 0.5 ** (recent["age_days"] / HALF_LIFE_DAYS)

buzz = recent.groupby("movieId").apply(
    lambda g: pd.Series({
        "weighted_count": g["decay"].sum(),
        "weighted_avg_rating": np.average(g["rating"], weights=g["decay"]),
    }),
    include_groups=False,
).reset_index()

C = buzz["weighted_count"].quantile(0.6)   # Bayesian prior strength
m = ratings["rating"].mean()               # prior mean (global average rating)
buzz["trend_score"] = (
    (buzz["weighted_count"] / (buzz["weighted_count"] + C)) * buzz["weighted_avg_rating"]
    + (C / (buzz["weighted_count"] + C)) * m
)

buzz = buzz.merge(movies, on="movieId")
buzz.sort_values("trend_score", ascending=False).head(10)[["title", "genres", "trend_score", "weighted_count"]]

## 4. Hybrid recommender

Combine the personalized predicted rating with the trending score (both min-max normalized to [0, 1]) into one ranking. `alpha` controls the mix:

- `alpha = 1.0` → pure personalization (ignore what's trending)
- `alpha = 0.0` → pure "what's hot this week" chart, same for every user
- Anything in between → personalized picks nudged toward what's currently popular

Movies the user has already rated are excluded. A brand-new user (not in the training data) has no personal signal, so the function automatically falls back to the trending chart — a standard cold-start handling strategy.

In [ ]:
def recommend_for_user(user_id, top_n=10, alpha=0.6):
    seen = set(ratings.loc[ratings.userId == user_id, "movieId"])
    candidates = buzz[~buzz["movieId"].isin(seen)].copy()

    if user_id in user_idx:
        uidx = user_idx[user_id]
        candidates["personal_score"] = candidates["movieId"].map(
            lambda m: pred_matrix[uidx, movie_idx[m]] if m in movie_idx else np.nan
        )
    else:
        candidates["personal_score"] = np.nan  # cold-start user

    def norm(s):
        return (s - s.min()) / (s.max() - s.min() + 1e-9)

    has_personal = candidates["personal_score"].notna().any()

    if not has_personal:
        candidates["final_score"] = norm(candidates["trend_score"])
    else:
        candidates["personal_norm"] = norm(candidates["personal_score"].fillna(candidates["personal_score"].mean()))
        candidates["trend_norm"] = norm(candidates["trend_score"])
        candidates["final_score"] = alpha * candidates["personal_norm"] + (1 - alpha) * candidates["trend_norm"]

    cols = ["title", "genres", "personal_score", "trend_score", "final_score"]
    return candidates.sort_values("final_score", ascending=False).head(top_n)[cols].reset_index(drop=True)

### Demo: recommendations for an existing user vs. a brand-new user

In [ ]:
recommend_for_user(user_id=1, top_n=10, alpha=0.6)

In [ ]:
# a user_id that doesn't exist in the training data -> no rating history -> falls back to the trending chart
recommend_for_user(user_id=999999, top_n=10)

## 5. Taking this to production

To turn this into something that actually reflects *this* week rather than a simulated one:

- **Live trending signal**: replace the `NOW`/`WINDOW_DAYS` block with a feed of real recent ratings or watch events (your own app's logs, or an API like TMDb's "trending this week" endpoint) and drop the simulation comment.
- **Retraining cadence**: retrain the SVD model periodically (e.g. nightly/weekly) as new ratings come in; it doesn't need to be real-time.
- **Availability filter**: intersect candidates with whatever's actually available to the user this week (streaming catalog, theater listings) before ranking, so you never recommend something they can't watch.
- **Cold start beyond new users**: the same fallback pattern (personal score → NaN → trending) also covers new movies with no ratings yet; give them a trending score based on early buzz (reviews, pre-release interest) instead of rating history.
- **Better factorization**: for a bigger dataset, swap `TruncatedSVD` on a dense matrix for a library built for sparse implicit/explicit feedback (e.g. `implicit`, or gradient-descent-based matrix factorization) — it scales far better than a dense user x item matrix.